# Graph Construction

In [1]:
# Load clean data
import numpy as np
import pandas as pd

conn = np.load('../data/conn_clean.npy')
df = pd.read_csv('../data/pheno_data_clean.csv')
ages = df['AGE_AT_SCAN'].values

print(f'Matrices: {conn.shape}')
print(f'Ages: {ages.shape}')

Matrices: (458, 200, 200)
Ages: (458,)


In [2]:
import torch
from torch_geometric.data import Data

def matrix_to_graph(matrix, age, threshold=0.3):
    """Converting a single subject's connectivity matrix into a PyG graph object."""
    
    num_nodes = matrix.shape[0]
    
    # Node features: each node's row from the matrix (its connectivity profile)
    node_features = torch.tensor(matrix, dtype=torch.float)
    
    # Getting upper triangle indices (excluding diagonal)
    row, col = np.triu_indices(num_nodes, k=1)
    
    # Getting the correlation values for those pairs
    weights = matrix[row, col]
    
    # Only keeping edges above the threshold
    mask = np.abs(weights) >= threshold
    row = row[mask]
    col = col[mask]
    weights = weights[mask]
    
    # Make edges bidirectional
    edge_index = torch.tensor(np.array([
        np.concatenate([row, col]),
        np.concatenate([col, row])
    ]), dtype=torch.long)
    
    # Edge weights (duplicated for both directions)
    edge_attr = torch.tensor(
        np.concatenate([weights, weights]), dtype=torch.float
    ).unsqueeze(1)
    
    # Target age
    y = torch.tensor([age], dtype=torch.float)
    
    return Data(x=node_features, edge_index=edge_index, edge_attr=edge_attr, y=y)

In [3]:
# Building datasets for different thresholds to compare later
thresholds = [0.0, 0.2, 0.3, 0.4]

datasets = {}
for t in thresholds:
    graphs = [matrix_to_graph(conn[i], ages[i], threshold=t) for i in range(len(ages))]
    datasets[t] = graphs
    print(f'Threshold {t}: {graphs[0].edge_index.shape[1]} edges (subject 1)')

Threshold 0.0: 39800 edges (subject 1)
Threshold 0.2: 17432 edges (subject 1)
Threshold 0.3: 9450 edges (subject 1)
Threshold 0.4: 4516 edges (subject 1)


In [4]:
from sklearn.model_selection import train_test_split

# Splitting indices once so all threshold versions use the same subjects
indices = list(range(len(ages)))
train_idx, temp_idx = train_test_split(indices, test_size=0.3, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

# Applying split to each threshold dataset
split_datasets = {}
for t in thresholds:
    split_datasets[t] = {
        'train': [datasets[t][i] for i in train_idx],
        'val': [datasets[t][i] for i in val_idx],
        'test': [datasets[t][i] for i in test_idx]
    }

print(f'Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}')

Train: 320 | Val: 69 | Test: 69


In [5]:
from torch_geometric.loader import DataLoader

# Creating DataLoaders for each threshold
loaders = {}
for t in thresholds:
    loaders[t] = {
        'train': DataLoader(split_datasets[t]['train'], batch_size=32, shuffle=True),
        'val': DataLoader(split_datasets[t]['val'], batch_size=32, shuffle = False),
        'test': DataLoader(split_datasets[t]['test'], batch_size=32, shuffle = False)
    }

print(f'DataLoaders created for thresholds: {thresholds}')

DataLoaders created for thresholds: [0.0, 0.2, 0.3, 0.4]


In [6]:
# Save everything needed for modeling
import pickle

save_data = {
    'split_datasets': split_datasets,
    'thresholds': thresholds,
    'train_idx': train_idx,
    'val_idx': val_idx,
    'test_idx': test_idx
}

with open('../data/processed_graphs.pkl', 'wb') as f:
    pickle.dump(save_data, f)

print('Saved.')

Saved.
